## MAN TruckScenes: selective download pipeline

**Purpose of this notebook:** downloads sensor data from the MAN TruckScenes S3 bucket to Google Drive, extracting only camera and radar keyframes (skipping LiDAR and sweeps, which make up ~95% of each zip).

**Strategy:** for each of the 7 sensor data zips (70–82 GB each):
1. Download zip to Colab local disk via aria2c
2. Selectively extract camera + radar keyframes to Drive
3. Delete the zip from local disk
4. Check Drive space before continuing to the next zip

**Result:** ~36 GB extracted (4 cameras + 6 radars, 23,902 keyframes per sensor) from ~520 GB of raw zips.

**Runtime:** select High-RAM in "Change runtime type" (no GPU needed, but the extra local disk space helps stage the 70–82 GB zips).

*Note: the download loop currently runs zips 4–7 only (`for i in range(4, 8)`), since zips 1–3 were downloaded in an earlier run with `NUM_ZIPS = 3`. To re-run all 7 from scratch, change back to `for i in range(1, NUM_ZIPS + 1)` (uncomment this part and comment/remove the other one).*

*Note2: In the final project we decided to not use the radar data. It is left in this pipeline as history with potential to extend this research further to the radar detection as well*

In [2]:
# mount google drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
# install aria2c
!apt-get -qq update
!apt-get -qq install -y aria2
!aria2c --version | head -1

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package libc-ares2:amd64.
(Reading database ... 118252 files and directories currently installed.)
Preparing to unpack .../libc-ares2_1.18.1-1ubuntu0.22.04.3_amd64.deb ...
Unpacking libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Selecting previously unselected package libaria2-0:amd64.
Preparing to unpack .../libaria2-0_1.36.0-1_amd64.deb ...
Unpacking libaria2-0:amd64 (1.36.0-1) ...
Selecting previously unselected package aria2.
Preparing to unpack .../aria2_1.36.0-1_amd64.deb ...
Unpacking aria2 (1.36.0-1) ...
Setting up libc-ares2:amd64 (1.18.1-1ubuntu0.22.04.3) ...
Setting up libaria2-0:amd64 (1.36.0-1) ...
Setting up aria2 (1.36.0-1) ...
Processing triggers for man-db (2.10.2-1) ...
Processing triggers for libc-bin (2.35-0ubuntu3.13) ...
/sbin/ldconfig.real: /usr/

In [ ]:
import shutil
import subprocess
import os
import json

In [ ]:
# where on google drive to store extracted data
GDRIVE_BASE = "/content/drive/MyDrive/truckscenes_raw"

# temp location for downloaded zips (using colab local disk - faster)
TMP_DIR = "/content/tmp_truckscenes"

# S3 base URL
S3_BASE = "https://man-truckscenes.s3.eu-central-1.amazonaws.com/release/trainval"

# dataset version
VERSION = "v1.2-trainval"

# sensor folders to extract from each zip
# 4 cameras + 6 radars
EXTRACT_PATTERNS = [
    "*/samples/CAMERA_LEFT_BACK/*",
    "*/samples/CAMERA_LEFT_FRONT/*",
    "*/samples/CAMERA_RIGHT_BACK/*",
    "*/samples/CAMERA_RIGHT_FRONT/*",
    "*/samples/RADAR_LEFT_BACK/*",
    "*/samples/RADAR_LEFT_FRONT/*",
    "*/samples/RADAR_LEFT_SIDE/*",
    "*/samples/RADAR_RIGHT_BACK/*",
    "*/samples/RADAR_RIGHT_FRONT/*",
    "*/samples/RADAR_RIGHT_SIDE/*",
]

# minimum free space on Drive (in GB) to continue downloading
MIN_FREE_GB = 5.0

# no. of sensor data zips
NUM_ZIPS = 7


In [ ]:
os.makedirs(GDRIVE_BASE, exist_ok=True)
os.makedirs(TMP_DIR, exist_ok=True)

In [ ]:
# helpers
def get_drive_free_gb():
  # get free (remaining) space on GD mount in GB
  total, used, free = shutil.disk_usage("/content/drive")
  return free / (1024 ** 3)

def get_colab_free_gb():
  # get remaining space on colab local disk in GB
  total, used, free = shutil.disk_usage("/content")
  return free / (1024 ** 3)

def print_storage_status():
  # print current storage status for both above
  drive_free = get_drive_free_gb()
  colab_free = get_colab_free_gb()

  print(f"Drive free: {drive_free:.2f} GB")
  print(f"Colab free: {colab_free:.2f} GB")
  print(f"Min required: {MIN_FREE_GB:.2f} GB")

  return drive_free, colab_free

def download_zip_to_colab(zip_name):
  # download zip from S3 directly to colab local disk
  url = f"{S3_BASE}/{zip_name}"
  dest = f"{TMP_DIR}/{zip_name}"

  if os.path.exists(dest):
        size_gb = os.path.getsize(dest) / (1024 ** 3)
        print(f"zip exists locally: {dest} ({size_gb:.2f} GB)")
        return dest

  cmd = [
        "aria2c",
        "-x", "8", "-s", "8", # 8 connections
        "-c", # continue partial
        "-d", TMP_DIR,
        "-o", zip_name,
        "--summary-interval=30",
        url,
    ]

  result = subprocess.run(cmd)

  if result.returncode != 0:
      print(f" ERROR. aria2c returned code {result.returncode}")
      return None

  if os.path.exists(dest):
      size_gb = os.path.getsize(dest) / (1024 ** 3)
      print(f" downloaded: {size_gb:.2f} GB")
      return dest
  else:
      print(f" ERROR. File not found after download")
      return None

def extract_selective(zip_path, extract_to):
    #extract only camera + radar keyframe folders from zip
    # build unzip command with all patterns
    cmd = ["unzip", "-q", "-o", zip_path, "-d", extract_to]
    cmd.extend(EXTRACT_PATTERNS)

    result = subprocess.run(cmd, capture_output=True, text=True)

    # unzip returns 0 on success, 1 on warnings (some patterns not found), 11 on no match
    if result.returncode > 1:
        print(f"  WARNING. unzip returned code {result.returncode}")
        if result.stderr:
            print(f"  stderr: {result.stderr[:500]}")

    # count extracted files
    count = 0
    for root, dirs, files in os.walk(extract_to):
        count += len(files)
    print(f" files in destination after extraction: {count}")

    return result.returncode <= 1

def delete_zip(zip_path):
   #delete downloaded zip to free space
    if os.path.exists(zip_path):
        size_gb = os.path.getsize(zip_path) / (1024 ** 3)
        os.remove(zip_path)

In [ ]:
# initially download metadata
meta_zip_name = f"man-truckscenes_metadata_{VERSION}.zip"
meta_local = f"{TMP_DIR}/{meta_zip_name}"
meta_extracted = os.path.join(GDRIVE_BASE, "man-truckscenes", "v1.2-trainval")

if os.path.exists(meta_extracted):
    print(f" already extracted on Drive, skipping.")
else:
    meta_url = f"{S3_BASE}/{meta_zip_name}"
    # this part: download metadata to local disk
    !aria2c -x 8 -s 8 -c -d "{TMP_DIR}" -o "{meta_zip_name}" "{meta_url}"

    # this part: extracting metadata to drive
    !unzip -q -o "{meta_local}" -d "{GDRIVE_BASE}"

    if os.path.exists(meta_local):
        os.remove(meta_local)



05/16 19:12:30 [NOTICE] Downloading 1 item(s)

05/16 19:12:42 [NOTICE] Download complete: /content/tmp_truckscenes/man-truckscenes_metadata_v1.2-trainval.zip

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
657212|OK  |    77MiB/s|/content/tmp_truckscenes/man-truckscenes_metadata_v1.2-trainval.zip

Status Legend:
(OK):download completed.


In [ ]:
# start downloading + extracting zips
results = {}

#for i in range(1, NUM_ZIPS + 1): (note: this was the initial run when the no. of zips was set to 3)
# those 3 are already downloaded, so we need the rest 4 of them
for i in range(4, 8):
    zip_name = f"man-truckscenes_sensordata{i:02d}_{VERSION}.zip"

    # check drive space before starting
    drive_free, colab_free = print_storage_status()

    if drive_free < MIN_FREE_GB:
        print(f" Drive has only {drive_free:.2f} GB free (min: {MIN_FREE_GB} GB)")
        results[zip_name] = "skipped (low Drive space)"
        break

    if colab_free < 20:
        print(f"  WARNING. Colab local disk low ({colab_free:.2f} GB). Zip might not fit but trying anyway.")

    # 1: download zip to colab local disk
    print(f"\n  [1/3] downloading to colab local disk")
    zip_path = download_zip_to_colab(zip_name)

    if zip_path is None:
        results[zip_name] = "FAILED (download error)"
        continue

    # 2: extract camera + radar to drive
    print(f"\n  [2/3] extracting camera + radar keyframes to drive")
    extract_ok = extract_selective(zip_path, GDRIVE_BASE)

    if not extract_ok:
        print(f"  WARNING: extraction had issues, but continuing")

    # 3: delete the zip from colab local disk
    print(f"\n  [3/3] cleaning up zip")
    delete_zip(zip_path)

    results[zip_name] = "OK" if extract_ok else "PARTIAL"


Drive free: 76.30 GB
Colab free: 114.79 GB
Min required: 5.00 GB

  [1/3] downloading to colab local disk
 downloaded: 74.39 GB

  [2/3] extracting camera + radar keyframes to drive
 files in destination after extraction: 134294

  [3/3] cleaning up zip
Drive free: 75.42 GB
Colab free: 115.84 GB
Min required: 5.00 GB

  [1/3] downloading to colab local disk
 downloaded: 69.89 GB

  [2/3] extracting camera + radar keyframes to drive
 files in destination after extraction: 167844

  [3/3] cleaning up zip
Drive free: 71.14 GB
Colab free: 114.23 GB
Min required: 5.00 GB

  [1/3] downloading to colab local disk
 downloaded: 77.46 GB

  [2/3] extracting camera + radar keyframes to drive
 files in destination after extraction: 201444

  [3/3] cleaning up zip
Drive free: 69.99 GB
Colab free: 111.97 GB
Min required: 5.00 GB

  [1/3] downloading to colab local disk
 downloaded: 81.89 GB

  [2/3] extracting camera + radar keyframes to drive
 files in destination after extraction: 239034

  [3/3] 

In [ ]:
# summary
for zip_name, status in results.items():
    print(f"{zip_name}: {status}")

base = os.path.join(GDRIVE_BASE, "man-truckscenes", "samples")
if os.path.exists(base):
    print()
    for sensor_dir in sorted(os.listdir(base)):
        sensor_path = os.path.join(base, sensor_dir)
        if os.path.isdir(sensor_path):
            n = len(os.listdir(sensor_path))
            total_bytes = sum(
                os.path.getsize(os.path.join(sensor_path, f))
                for f in os.listdir(sensor_path)
                if os.path.isfile(os.path.join(sensor_path, f))
            )
            gb = total_bytes / (1024 ** 3)
            print(f"  {sensor_dir}: {n} files, {gb:.2f} GB")

print_storage_status()

man-truckscenes_sensordata04_v1.2-trainval.zip: OK
man-truckscenes_sensordata05_v1.2-trainval.zip: OK
man-truckscenes_sensordata06_v1.2-trainval.zip: OK
man-truckscenes_sensordata07_v1.2-trainval.zip: OK

  CAMERA_LEFT_BACK: 23902 files, 8.13 GB
  CAMERA_LEFT_FRONT: 23902 files, 8.12 GB
  CAMERA_RIGHT_BACK: 23902 files, 9.69 GB
  CAMERA_RIGHT_FRONT: 23902 files, 8.53 GB
  RADAR_LEFT_BACK: 23902 files, 0.33 GB
  RADAR_LEFT_FRONT: 23902 files, 0.34 GB
  RADAR_LEFT_SIDE: 23902 files, 0.29 GB
  RADAR_RIGHT_BACK: 23902 files, 0.29 GB
  RADAR_RIGHT_FRONT: 23902 files, 0.33 GB
  RADAR_RIGHT_SIDE: 23902 files, 0.24 GB
Drive free: 65.35 GB
Colab free: 111.93 GB
Min required: 5.00 GB


(65.3450698852539, 111.93038940429688)

In [ ]:
# metadata check
BASE = "/content/drive/MyDrive/truckscenes_raw/man-truckscenes"

for f in ["sample.json", "scene.json", "sample_data.json",
          "sample_annotation.json", "category.json",
          "calibrated_sensor.json", "sensor.json"]:
    path = os.path.join(BASE, "v1.2-trainval", f)
    if not os.path.exists(path):
        print(f"missing: {f}")

In [ ]:
# download the test annotation metadata
!aria2c -x 8 -s 8 -c \
  -d /content/tmp_truckscenes \
  -o man-truckscenes_metadata_v1.2-test.zip \
  "https://man-truckscenes.s3.eu-central-1.amazonaws.com/release/test/man-truckscenes_metadata_v1.2-test.zip"

!unzip -q -o /content/tmp_truckscenes/man-truckscenes_metadata_v1.2-test.zip \
  -d /content/drive/MyDrive/truckscenes_raw

!rm /content/tmp_truckscenes/man-truckscenes_metadata_v1.2-test.zip


05/18 23:37:39 [NOTICE] Downloading 1 item(s)

05/18 23:37:42 [NOTICE] Download complete: /content/tmp_truckscenes/man-truckscenes_metadata_v1.2-test.zip

Download Results:
gid   |stat|avg speed  |path/URI
======+====+===========+=======================================================
1d5130|OK  |   107MiB/s|/content/tmp_truckscenes/man-truckscenes_metadata_v1.2-test.zip

Status Legend:
(OK):download completed.


In [ ]:
# check whether annotations exist for the test set
path = "/content/drive/MyDrive/truckscenes_raw/man-truckscenes/v1.2-test/sample_annotation.json"

if os.path.exists(path):
    with open(path) as f:
        annotations = json.load(f)
    print(f"annotations found: {len(annotations)}")
else:
    print("test metadata not downloaded yet")

annotations found: 0
